# Notebook 06 — Regime Detection

**Phase 2 · Strategy Modules (3 / 4)**

---

## 🎯 Learning Objectives

| # | Objective |
|---|----------|
| 1 | Understand *why* regime detection is the backbone of a regime-adaptive bot |
| 2 | Classify markets into **bull / ranging / bear** using EMA crossover + volatility |
| 3 | Implement an **anti-whipsaw confirmation filter** to reduce false regime switches |
| 4 | Visualise regime history with colour-coded price charts |
| 5 | Compare scratch code to the production `classify_regime_history()` |

### Prerequisites
- NB02 (EMA)
- NB04 & NB05 (momentum + mean reversion — to understand *why* regime matters)

In [ ]:
# ── Boilerplate ────────────────────────────────────────────
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = str(pathlib.Path.cwd().resolve().parents[1])
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates

plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
print("✅ imports ready  |  project root:", ROOT)

---
## 1 · Why Regime Detection?

No single strategy works in all market conditions:

| Regime | Best Strategy | Worst Strategy |
|--------|--------------|----------------|
| **Bull** 📈 | Momentum (ride the trend) | Mean reversion (fights the trend) |
| **Ranging** ↔️ | Mean reversion (fade extremes) | Momentum (whipsawed) |
| **Bear** 📉 | Defensive / Cash | Everything (but especially momentum) |

Our bot uses regime detection to **re-weight** the four sub-strategies dynamically.  The regime detector is the **single most important decision** in the pipeline.

### Detection Logic (from `config/strategy_params.yaml`)

```yaml
regime:
  ema_fast_period: 20
  ema_slow_period: 50
  volatility_lookback: 14
  volatility_threshold_multiplier: 1.5
  confirmation_periods: 2
```

---
## 2 · Synthetic Price Data with Multiple Regimes

In [ ]:
np.random.seed(42)
DAYS = 300
dates = pd.date_range(end=pd.Timestamp.now().normalize(), periods=DAYS, freq="D")

# Construct a price series with clear regime phases
drift = np.zeros(DAYS)
vol   = np.zeros(DAYS)
true_regime = []

# Phase 1: Bull  (days 0–80)
drift[0:80]   = 0.003;  vol[0:80]   = 0.015
true_regime += ["bull"] * 80
# Phase 2: Ranging (days 80–160)
drift[80:160]  = 0.000;  vol[80:160]  = 0.020
true_regime += ["ranging"] * 80
# Phase 3: Bear (days 160–220)
drift[160:220] = -0.004; vol[160:220] = 0.035
true_regime += ["bear"] * 60
# Phase 4: Recovery Bull (days 220–300)
drift[220:300] = 0.004;  vol[220:300] = 0.018
true_regime += ["bull"] * 80

log_returns = np.array([np.random.normal(d, v) for d, v in zip(drift, vol)])
prices = pd.Series(60000 * np.exp(np.cumsum(log_returns)), index=dates, name="BTCUSDT")

true_regime_series = pd.Series(true_regime, index=dates, name="true_regime")

print(f"Price range: ${prices.min():,.0f} – ${prices.max():,.0f}")
print(f"True regimes: {dict(true_regime_series.value_counts())}")

---
## 3 · Step 1: EMA Crossover

The first signal is the relationship between a **fast EMA** (20-day) and a **slow EMA** (50-day):

- **Fast > Slow** → uptrend bias (bull)
- **Fast < Slow** → downtrend bias (bear / ranging)

In [ ]:
EMA_FAST = 20   # from strategy_params.yaml
EMA_SLOW = 50

ema_fast = prices.ewm(span=EMA_FAST, adjust=False).mean()
ema_slow = prices.ewm(span=EMA_SLOW, adjust=False).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(prices.index, prices, label="Price", lw=1, alpha=0.7)
ax.plot(ema_fast.index, ema_fast, label=f"EMA-{EMA_FAST}", lw=1.5)
ax.plot(ema_slow.index, ema_slow, label=f"EMA-{EMA_SLOW}", lw=1.5)

# Shade bull (fast > slow) vs bear (fast < slow)
bull_mask = ema_fast > ema_slow
ax.fill_between(prices.index, prices.min() * 0.9, prices.max() * 1.1,
                where=bull_mask, alpha=0.05, color="green", label="Fast > Slow")
ax.fill_between(prices.index, prices.min() * 0.9, prices.max() * 1.1,
                where=~bull_mask, alpha=0.05, color="red", label="Fast < Slow")
ax.legend(fontsize=8)
ax.set_title("EMA Crossover Signal")
ax.set_ylabel("Price")
plt.tight_layout()
plt.show()

---
## 4 · Step 2: Volatility Threshold

High volatility overrides an EMA-based bull call → forces **bear** classification.

$$
\text{vol\_threshold} = \sigma_{\text{baseline}}(60\text{d}) \times 1.5
$$

If current 14-day realised vol exceeds this threshold, the regime flips to **bear** even if EMAs are bullish.

In [ ]:
VOL_LOOKBACK = 14
VOL_BASELINE = 60
VOL_MULTIPLIER = 1.5

returns = prices.pct_change()
rolling_vol = returns.rolling(VOL_LOOKBACK).std(ddof=0)
baseline_vol = returns.rolling(VOL_BASELINE).std(ddof=0)
vol_threshold = baseline_vol * VOL_MULTIPLIER

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(rolling_vol.index, rolling_vol, label=f"{VOL_LOOKBACK}d Realised Vol", lw=1.2)
ax.plot(vol_threshold.index, vol_threshold, label=f"Threshold ({VOL_MULTIPLIER}× baseline)",
        lw=1.2, ls="--", color="red")
high_vol = rolling_vol > vol_threshold
ax.fill_between(rolling_vol.index, 0, rolling_vol.max() * 1.2,
                where=high_vol, alpha=0.15, color="red", label="Elevated Volatility")
ax.set_ylabel("Volatility")
ax.set_title("Volatility vs Adaptive Threshold")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## 5 · Step 3: Combine → Base Regime

The `detect_regime()` function combines EMA + volatility into a single classification:

In [ ]:
def detect_regime(price, ema_f, ema_s, vol, vol_thresh):
    """Classify a single bar into bull / ranging / bear."""
    # Strong bull: price above both EMAs AND low vol
    if price > ema_f > ema_s and vol <= vol_thresh:
        return "bull"
    # Strong bear: price below both EMAs
    if price < ema_f < ema_s:
        return "bear"
    # High vol override → bear
    if vol > vol_thresh:
        return "bear"
    # EMA bullish alignment → bull
    if ema_f > ema_s:
        return "bull"
    # Default
    return "ranging"

# Classify every bar
base_regimes = []
for ts in prices.index:
    if any(pd.isna(x) for x in [ema_fast[ts], ema_slow[ts], rolling_vol[ts], vol_threshold[ts]]):
        base_regimes.append("unknown")
    else:
        base_regimes.append(detect_regime(
            float(prices[ts]), float(ema_fast[ts]), float(ema_slow[ts]),
            float(rolling_vol[ts]), float(vol_threshold[ts]),
        ))

base_regime_series = pd.Series(base_regimes, index=prices.index)
print("Base regime counts:")
print(base_regime_series.value_counts())

---
## 6 · Step 4: Anti-Whipsaw Confirmation Filter

The base regime flips frequently — this causes **whipsaw** (rapid back-and-forth switches that trigger unnecessary rebalances).  The **confirmation filter** requires a regime to persist for `confirmation_periods` bars before it becomes the **active regime**.

```
Day:    1    2    3    4    5    6    7    8
Base:   bull bull bear bear bear bull bull bull
Active: bull bull bull bull bear  bear bear bull  ← lag from confirmation
```

With `confirmation_periods=2`, a new regime must appear for 2 consecutive bars before the active regime switches.

In [ ]:
CONFIRMATION = 2  # from strategy_params.yaml

def apply_confirmation(base_regimes: list[str], confirmation_periods: int) -> list[str]:
    """Anti-whipsaw: only change active regime after N consecutive confirmations."""
    active_regimes = []
    last_active = "unknown"
    streak_regime = "unknown"
    streak_count  = 0
    
    for regime in base_regimes:
        if regime == "unknown":
            active_regimes.append(last_active)
            continue
        
        # Track streak
        if regime == streak_regime:
            streak_count += 1
        else:
            streak_regime = regime
            streak_count = 1
        
        # First non-unknown
        if last_active == "unknown":
            last_active = regime
        elif regime != last_active and streak_count >= max(1, confirmation_periods):
            last_active = regime
        
        active_regimes.append(last_active)
    
    return active_regimes

active_regimes = apply_confirmation(base_regimes, CONFIRMATION)
active_regime_series = pd.Series(active_regimes, index=prices.index)

# Compare base vs active
switches_base   = (base_regime_series != base_regime_series.shift(1)).sum()
switches_active = (active_regime_series != active_regime_series.shift(1)).sum()
print(f"Base regime switches:   {switches_base}")
print(f"Active regime switches: {switches_active}")
print(f"Reduction:              {switches_base - switches_active} fewer switches ({(1 - switches_active/switches_base)*100:.0f}%)")

---
## 7 · Visualisation: Regime-Coloured Price Chart

In [ ]:
REGIME_COLOURS = {"bull": "#2ecc71", "ranging": "#f39c12", "bear": "#e74c3c", "unknown": "#95a5a6"}

def plot_regime_chart(prices, regimes, title):
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(prices.index, prices, color="black", lw=1, alpha=0.7)
    
    # Colour background by regime
    prev_regime = regimes.iloc[0]
    start = prices.index[0]
    for i in range(1, len(regimes)):
        if regimes.iloc[i] != prev_regime or i == len(regimes) - 1:
            end = prices.index[i]
            ax.axvspan(start, end, alpha=0.2, color=REGIME_COLOURS.get(prev_regime, "gray"))
            start = end
            prev_regime = regimes.iloc[i]
    
    ax.set_title(title)
    ax.set_ylabel("Price")
    # Legend
    patches = [mpatches.Patch(color=c, alpha=0.3, label=r.title()) for r, c in REGIME_COLOURS.items() if r != "unknown"]
    ax.legend(handles=patches, loc="upper left")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    plt.tight_layout()
    plt.show()

plot_regime_chart(prices, active_regime_series, "Detected Regimes (with confirmation filter)")

In [ ]:
# Side-by-side: base vs active vs ground truth
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for ax, series, title in [
    (axes[0], true_regime_series, "Ground Truth Regime"),
    (axes[1], base_regime_series, "Base Regime (no confirmation)"),
    (axes[2], active_regime_series, "Active Regime (confirmation=2)"),
]:
    ax.plot(prices.index, prices, color="black", lw=0.8, alpha=0.6)
    prev_r = series.iloc[0]
    start = prices.index[0]
    for i in range(1, len(series)):
        if series.iloc[i] != prev_r or i == len(series) - 1:
            end = prices.index[i]
            ax.axvspan(start, end, alpha=0.25, color=REGIME_COLOURS.get(prev_r, "gray"))
            start = end
            prev_r = series.iloc[i]
    ax.set_title(title)
    ax.set_ylabel("Price")

patches = [mpatches.Patch(color=c, alpha=0.3, label=r.title()) for r, c in REGIME_COLOURS.items() if r != "unknown"]
axes[0].legend(handles=patches, loc="upper left", fontsize=8)
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.tight_layout()
plt.show()

---
## 8 · Regime Accuracy Analysis

In [ ]:
# Only compare where both are known
mask = (active_regime_series != "unknown") & (true_regime_series != "unknown")
compare = pd.DataFrame({
    "true": true_regime_series[mask],
    "detected": active_regime_series[mask],
})
compare["match"] = compare["true"] == compare["detected"]
accuracy = compare["match"].mean()
print(f"Overall accuracy: {accuracy:.1%}\n")

# Per-regime accuracy
for regime in ["bull", "ranging", "bear"]:
    subset = compare[compare["true"] == regime]
    if len(subset) > 0:
        acc = subset["match"].mean()
        print(f"  {regime:8s}: {acc:.1%}  ({len(subset)} bars)")

# Confusion matrix
print("\nConfusion matrix:")
print(pd.crosstab(compare["true"], compare["detected"], margins=True))

---
## 9 · Production Code: `classify_regime_history()`

In [ ]:
from bot.strategy.regime_detector import classify_regime_history, detect_regime as prod_detect

prod_df = classify_regime_history(
    prices,
    ema_fast_period=20,
    ema_slow_period=50,
    volatility_lookback=14,
    volatility_baseline_period=60,
    volatility_threshold_multiplier=1.5,
    confirmation_periods=2,
)

print("Production output columns:", prod_df.columns.tolist())
prod_df[["price", "ema_fast", "ema_slow", "volatility", "base_regime", "active_regime"]].tail(5)

In [ ]:
# Verify our scratch matches production
prod_active = prod_df["active_regime"]
match_rate = (prod_active == active_regime_series).mean()
print(f"Scratch vs Production agreement: {match_rate:.1%}")

# Show the production regime chart
plot_regime_chart(prices, prod_active, "Production Regime Detection")

---
## 10 · Regime Impact on Strategy Selection

This is the bridge to NB07 (Ensemble).  The regime determines the **weight** each sub-strategy receives:

```python
# From bot/strategy/ensemble.py
_REGIME_WEIGHTS = {
    "bull":    {"momentum": 0.5, "mean_reversion": 0.1, "pairs": 0.2, "sector": 0.2},
    "ranging": {"momentum": 0.2, "mean_reversion": 0.4, "pairs": 0.3, "sector": 0.1},
    "bear":    {"momentum": 0.1, "mean_reversion": 0.3, "pairs": 0.3, "sector": 0.3},
}
```

In [ ]:
REGIME_WEIGHTS = {
    "bull":    {"momentum": 0.5, "mean_reversion": 0.1, "pairs": 0.2, "sector": 0.2},
    "ranging": {"momentum": 0.2, "mean_reversion": 0.4, "pairs": 0.3, "sector": 0.1},
    "bear":    {"momentum": 0.1, "mean_reversion": 0.3, "pairs": 0.3, "sector": 0.3},
}

weight_df = pd.DataFrame(REGIME_WEIGHTS).T
weight_df.plot.bar(figsize=(10, 5), rot=0, width=0.7,
                   color=["#3498db", "#e74c3c", "#f39c12", "#2ecc71"])
plt.title("Sub-Strategy Weights by Regime")
plt.ylabel("Weight")
plt.xlabel("Regime")
plt.legend(title="Strategy")
plt.tight_layout()
plt.show()

---
## 11 · Sensitivity: Confirmation Period

How does `confirmation_periods` affect switch frequency?

In [ ]:
results = []
for cp in range(1, 8):
    active = apply_confirmation(base_regimes, cp)
    active_s = pd.Series(active, index=prices.index)
    n_switches = (active_s != active_s.shift(1)).sum()
    # Accuracy vs ground truth
    m = (active_s != "unknown") & (true_regime_series != "unknown")
    acc = (active_s[m] == true_regime_series[m]).mean()
    results.append({"confirmation": cp, "switches": n_switches, "accuracy": acc})

res_df = pd.DataFrame(results).set_index("confirmation")

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.bar(res_df.index, res_df["switches"], alpha=0.6, color="#3498db", label="Switches")
ax2.plot(res_df.index, res_df["accuracy"], color="#e74c3c", marker="o", lw=2, label="Accuracy")
ax1.set_xlabel("Confirmation Periods")
ax1.set_ylabel("Number of Switches", color="#3498db")
ax2.set_ylabel("Accuracy vs Ground Truth", color="#e74c3c")
ax1.set_title("Confirmation Period: Stability vs Accuracy Trade-off")
fig.legend(loc="upper right", bbox_to_anchor=(0.88, 0.88))
plt.tight_layout()
plt.show()

res_df.style.format({"accuracy": "{:.1%}"})

---
## 12 · Key Takeaways

| Concept | Detail |
|---------|--------|
| **EMA crossover** | Fast(20) vs Slow(50) — primary trend signal |
| **Volatility override** | 14d vol > 1.5× baseline(60d) → bear |
| **Confirmation filter** | Must persist for 2 bars to switch active regime |
| **Three regimes** | bull / ranging / bear |
| **Downstream effect** | Regime → weight allocation for 4 sub-strategies |

### Regime Detection Pipeline

```
Price Series
  │
  ├── EMA Fast (20) & EMA Slow (50)
  ├── 14d Realised Volatility
  ├── 60d Baseline Vol × 1.5 threshold
  │
  ├── detect_regime() → base_regime (bar-by-bar)
  │
  └── apply_confirmation(2) → active_regime (anti-whipsaw)
```

---
## 🔬 Exercises

1. **EMA period sweep:** Try fast/slow combos of (10, 30), (20, 50), and (30, 100). Which gives the best accuracy on the synthetic data?

2. **Volatility multiplier:** Change `vol_threshold_multiplier` from 1.0 to 2.5 in steps of 0.25. At what level does the bear regime almost disappear?

3. **Hidden Markov Model:** Replace EMA crossover with a 3-state Hidden Markov Model (use `hmmlearn`). Does it detect regimes earlier at the cost of more false positives?

4. **Regime transitions:** Plot a transition matrix: given the current active regime, what is the probability of each next regime? Are transitions symmetric?

---
## ✅ Knowledge Check

1. Why does the bot use *two* EMAs instead of a single one?
2. What happens if volatility spikes during a bull trend (price > EMA_fast > EMA_slow)?
3. How does the confirmation filter reduce whipsaw? What is the trade-off?
4. In which regime does momentum get the highest weight? Why?
5. How would you add a fourth regime (e.g., "high-vol bull")?

---
## 🔗 Next

**[NB07 — Ensemble Strategy & Sentiment →](07_Ensemble_Strategy_and_Sentiment.ipynb)**

We'll bring everything together: combine momentum, mean reversion, pairs, and sector rotation using regime-weighted blending, with a sentiment overlay from the Fear & Greed Index.